# VLM 파싱 모델 비교 (QWEN3 vs OpenAI)



## 1. 구글 드라이버 연동

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import os

FILE_DIR = "/content/drive/MyDrive/3team_project"
OUTPUT_DIR = "/content/drive/MyDrive/3team_project/vlm_parsed"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"OUTPUT 경로 폴더 생성 : {OUTPUT_DIR}")

OUTPUT 경로 폴더 생성 : /content/drive/MyDrive/3team_project/vlm_parsed


## 2. 패키지 설치

In [4]:
!pip install -q pymupdf Pillow tqdm pandas numpy
!pip install -q openai
!pip install -q transformers>=4.51.0 accelerate qwen-vl-utils
!pip install -q bitsandbytes # INT4 양자화용
!pip install -q langchain langchain-experimental langchain-openai
!pip install -q rouge-score jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 65

각 패키지의 용도는 다음과 같습니다:

- `pymupdf`, `Pillow`: 문서를 이미지로 변환하고 처리합니다.
(HWP 파일은 보통 PDF로 변환 후, 이 라이브러리들을 통해 이미지로 만들어 AI에게 보여줍니다.)
- `openai`: OpenAI의 GPT-4o(Vision) 모델을 사용하기 위한 라이브러리입니다.
- `transformers`, `qwen-vl-utils`: Qwen2-VL (Qwen3라고 지칭하신 최신 모델 등)을 로컬에서 불러오고 처리하는 데 필요합니다.
- `accelerate`, `bitsandbytes`: Colab GPU 메모리 한계 내에서 거대 모델(Qwen)을 효율적으로 돌리기 위한 최적화(양자화) 도구입니다.
- `langchain` 관련: AI 모델과 데이터를 쉽게 연결해주는 프레임워크입니다.
- `rouge-score`, `jiwer`: 파싱 결과가 얼마나 정확한지 평가(채점)하는 도구입니다.

참고: HWP 파일을 직접 읽는 도구는 포함되어 있지 않습니다.
보통 HWP → PDF 변환 후, PDF를 pymupdf로 이미지화하여 Qwen/OpenAI에게 시각적으로 읽게 하는 방식을 사용합니다.

## 3. 모델 로드

허깅페이스에서 Qwen3-VL-8B 모델을 로드합니다.

In [6]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

# 모델 불러오기
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
model = Qwen3VLForConditionalGeneration.from_pretrained(
   MODEL_ID , dtype="auto", device_map ="auto"
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f'모델 로드 완료 : {MODEL_ID}')

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

모델 로드 완료 : Qwen/Qwen3-VL-8B-Instruct


## 4. hwp->PDF 이미지 변환

In [ ]:
import subprocess

def convert_hwp_to_pdf(source_dir, output_dir):
  """
  지정된 폴더(source_dir) 내의 모든 .hwp 파일을 찾아 PDF로 변환합니다.
  """
  if not os.path.exists(output_dir):
    os.makedirs(output_dir)
  # 소스 디렉토리 내의 모든 파일을 확인
  if not os.path.exists(source_dir):
    print(f"경로를 찾을 수 없습니다 : {source_dir}")
    return

  all_files = os.listdir(source_dir)

  # .hwp 확장자만 가진 파일만 골라냅니다.
  hwp_files =[ f for f in all_files if f.endswith('.hwp')]
  print(f"발견된 HWP 파일 갯수 : {len(hwp_files)}")


  command = [
      "libreoffice",
      "--headless",
      "--convert-to", "pdf",
      FILE_DIR,
      "--outdir", output_dir
  ]

  try:
    # 명령어 실행
    result = subprocess.run(command, capture_output=True, text=True, check=True)

    # 변환된 파일 경로 확인 (.hwp-> pdf)
    full_filename = os.path.basename(FILE_DIR)
    file_name_only = os.path.splitext(full_filename)[0]
    pdf_path = os.path.join(OUTPUT_DIR, f"{file_name_only}.pdf")

    if os.path.exists(pdf_path):
      print(f"변환 완료 : {pdf_path}")
      return pdf_path
    return None

  except subprocess.CalledProcessError as e:
    print(f"변환 실패 : {e.stderr}")
    return None

In [ ]:
import os
import subprocess

def convert_hwp_to_pdf_batch(source_dir, output_dir):
    """
    지정된 폴더(source_dir) 내의 모든 .hwp 파일을 찾아 PDF로 변환합니다.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)

    # 1. 소스 디렉토리 내의 모든 파일을 확인합니다.
    if not os.path.exists(source_dir):
        print(f"경로를 찾을 수 없습니다: {source_dir}")
        return

    all_files = os.listdir(source_dir)

    # 2. .hwp 확장자를 가진 파일만 골라냅니다.
    hwp_files = [f for f in all_files if f.endswith('.hwp')]
    print(f"발견된 HWP 파일 개수: {len(hwp_files)}")

    # 3. 각 HWP 파일에 대해 변환 명령을 실행합니다.
    for file_name in hwp_files:
        input_path = os.path.join(source_dir, file_name)

        print(f"변환 중: {file_name} ...")

        command = [
            "libreoffice",
            "--headless",
            "--convert-to", "pdf",
            input_path,   # 폴더가 아닌 개별 파일 경로를 넣어야 합니다.
            "--outdir", output_dir
        ]

        try:
            subprocess.run(command, capture_output=True, check=True)

            # 변환 결과 확인 로직
            pdf_filename = os.path.splitext(file_name)[0] + ".pdf"
            pdf_path = os.path.join(output_dir, pdf_filename)

            if os.path.exists(pdf_path):
                print(f" -> 성공: {pdf_path}")
            else:
                print(f" -> 실패 (파일이 생성되지 않음): {file_name}")

        except subprocess.CalledProcessError as e:
            print(f" -> 에러 발생: {e}")

### 4-1 LibreOffice 변환 명령어 구성:
1. "libreoffice": 리브레오피스 프로그램 실행
2. "--headless": GUI 없이 백그라운드에서 실행 (서버/Colab 환경 필수)
3. "--convert-to", "pdf": 입력 파일을 PDF로 변환
4. FILE_DIR: 변환할 대상 파일의 경로
5. "--outdir", OUTPUT_DIR: 변환된 결과물이 저장될 폴더 지정

### 4-2. 외부 프로세스 제어 (subprocess)

VLM 파싱 과정에서 외부 도구(LibreOffice 등)를 실행하기 위해 `subprocess` 모듈을 사용합니다.

### 핵심 요약
*   **목적:** 파이썬 코드 내에서 리눅스 명령어를 실행하고 결과를 제어.
*   **메서드:** `subprocess.run()` (Python 3.5+ 권장)
*   **주요 파라미터:**
    *   `args`: 실행할 명령어 리스트 (예: `["ls", "-l"]`)
    *   `capture_output=True`: 실행 결과를 변수에 담음.
    *   `text=True`: 결과를 문자열로 처리.
    *   `check=True`: 명령어 실패 시 예외 발생.
    *   `shell=True`: 쉘을 통해 실행합니다. (보안상 주의 필요)
    *   `cwd` : 명령어를 실행할 작업 디렉토리를 지정합니다.